##  PART 1: DATA PIPELINE - PubMed API

**Purpose:** Fetch 5 most recent articles for each medical term from PubMed API.

In [ ]:
!pip install requests pandas tqdm tenacity

In [ ]:
import requests
import time
import json
import pandas as pd
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import xml.etree.ElementTree as ET
from collections import defaultdict
print("✅ All libraries imported")


In [ ]:
# PubMed API endpoints
BASE_SEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
BASE_FETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(requests.exceptions.RequestException)
)
def search_pubmed(term, retmax=5):
    """
    Search PubMed for a term and return list of PMIDs.

    Args:
        term: Search query string
        retmax: Maximum number of results to return

    Returns:
        List of PMID strings
    """
    params = {
        "db": "pubmed",
        "term": term,
        "retmax": retmax,
        "sort": "relevance",
        "retmode": "json"
    }
    response = requests.get(BASE_SEARCH_URL, params=params)
    response.raise_for_status()
    data = response.json()
    return data["esearchresult"]["idlist"]

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10)
)
def fetch_abstracts(pmids):
    """
    Fetch full article data in XML format for given PMIDs.

    Args:
        pmids: List of PubMed IDs

    Returns:
        XML string containing article data
    """
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml"
    }
    response = requests.get(BASE_FETCH_URL, params=params)
    response.raise_for_status()
    return response.text

In [ ]:
def parse_pubmed_article(xml_str, pmid, query_term):
    """
    Extract relevant fields from PubMed XML.

    Args:
        xml_str: XML string for a single article
        pmid: PubMed ID
        query_term: Original search term that found this article

    Returns:
        Dictionary with article metadata
    """
    root = ET.fromstring(xml_str)
    article = root.find(".//PubmedArticle")
    if article is None:
        return None

    # Extract title
    title_el = article.find(".//ArticleTitle")
    title = title_el.text if title_el is not None else ""

    # Extract abstract
    abstract_el = article.find(".//Abstract/AbstractText")
    abstract = abstract_el.text if abstract_el is not None else ""

    # Extract first author
    author_el = article.find(".//AuthorList/Author")
    if author_el is not None:
        last = author_el.find("LastName")
        first = author_el.find("ForeName")
        author = f"{first.text if first is not None else ''} {last.text if last is not None else ''}".strip()
    else:
        author = ""

    # Extract journal
    journal_el = article.find(".//Journal/Title")
    journal = journal_el.text if journal_el is not None else ""

    # Extract year
    pubdate_el = article.find(".//PubDate/Year")
    year = pubdate_el.text if pubdate_el is not None else ""

    # Extract DOI
    doi_el = article.find(".//ArticleId[@IdType='doi']")
    doi = doi_el.text if doi_el is not None else ""

    return {
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "first_author": author,
        "journal": journal,
        "year": year,
        "doi": doi,
        "query_terms": [query_term]
    }

In [ ]:
def build_corpus(csv_path="medical_terms.csv"):
    """
    Main pipeline: fetch articles for all terms, deduplicate, return corpus.

    Args:
        csv_path: Path to medical_terms.csv file

    Returns:
        Tuple of (corpus_list, errors_list, terms_list)
    """
    # Read CSV file
    df = pd.read_csv(csv_path)
    terms = df["term"].tolist()

    # Store unique articles by PMID
    all_articles = {}
    errors = []

    # Rate limiting: max 3 requests per second
    rate_limit_delay = 1.0 / 3

    for term in tqdm(terms, desc="Processing terms"):
        try:
            # Step 1: Search for PMIDs
            pmids = search_pubmed(term, retmax=5)
            time.sleep(rate_limit_delay)

            if not pmids:
                continue

            # Step 2: Fetch article data
            xml_data = fetch_abstracts(pmids)
            time.sleep(rate_limit_delay)

            # Step 3: Parse each article
            root = ET.fromstring(xml_data)
            for pmid in pmids:
                # Find article block for this PMID
                article_xml = root.find(f".//PubmedArticle[.//PMID[text()='{pmid}']]")
                if article_xml is None:
                    continue

                parsed = parse_pubmed_article(ET.tostring(article_xml), pmid, term)
                if parsed:
                    if pmid in all_articles:
                        # Article exists, add query term if not already present
                        if term not in all_articles[pmid]["query_terms"]:
                            all_articles[pmid]["query_terms"].append(term)
                    else:
                        all_articles[pmid] = parsed

        except Exception as e:
            errors.append({"term": term, "error": str(e)})

    return list(all_articles.values()), errors, terms

In [ ]:
# Create medical_terms.csv directly in Colab
csv_content = """term
atrial fibrillation
type 2 diabetes mellitus
pediatric asthma management
acute otitis media
chronic kidney disease
iron deficiency anemia
community acquired pneumonia
gestational diabetes
celiac disease diagnosis
allergic rhinitis treatment"""

# Write to file
with open("medical_terms.csv", "w", encoding="utf-8") as f:
    f.write(csv_content)

print("✅ medical_terms.csv created in Colab!")

# Verify
import pandas as pd
df = pd.read_csv("medical_terms.csv")
print(df)

In [ ]:


def parse_pubmed_article_v2(xml_str, pmid, query_term):
    """
    Extract relevant fields from PubMed XML - Fixed version.
    """
    try:
        root = ET.fromstring(xml_str)

        # Try to find article directly - handle namespaces
        article = None
        for elem in root.iter():
            if elem.tag.endswith('PubmedArticle'):
                # Check if this article has our PMID
                pmid_elem = elem.find('.//PMID')
                if pmid_elem is not None and pmid_elem.text == pmid:
                    article = elem
                    break

        if article is None:
            return None

        # Extract title
        title_el = article.find('.//ArticleTitle')
        title = title_el.text if title_el is not None else ""

        # Extract abstract (handle multiple AbstractText elements)
        abstract_texts = []
        for abs_elem in article.findall('.//AbstractText'):
            if abs_elem.text:
                abstract_texts.append(abs_elem.text)
        abstract = " ".join(abstract_texts)

        # Extract first author
        author_el = article.find('.//AuthorList/Author')
        if author_el is not None:
            last = author_el.find('LastName')
            first = author_el.find('ForeName')
            author = f"{first.text if first is not None else ''} {last.text if last is not None else ''}".strip()
        else:
            author = ""

        # Extract journal
        journal_el = article.find('.//Journal/Title')
        journal = journal_el.text if journal_el is not None else ""

        # Extract year
        pubdate_el = article.find('.//PubDate/Year')
        year = pubdate_el.text if pubdate_el is not None else ""

        # Extract DOI
        doi_el = article.find(".//ArticleId[@IdType='doi']")
        doi = doi_el.text if doi_el is not None else ""

        return {
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "first_author": author,
            "journal": journal,
            "year": year,
            "doi": doi,
            "query_terms": [query_term]
        }
    except Exception as e:
        print(f"Parse error for PMID {pmid}: {e}")
        return None


def build_corpus_v2(csv_path="medical_terms.csv"):
    """
    Main pipeline: fetch articles for all terms, deduplicate, return corpus.
    Fixed version with better XML parsing.
    """
    # Read CSV file
    df = pd.read_csv(csv_path)
    terms = df["term"].tolist()

    # Store unique articles by PMID
    all_articles = {}
    errors = []

    # Rate limiting: max 3 requests per second
    rate_limit_delay = 1.0 / 3

    for term in tqdm(terms, desc="Processing terms"):
        try:
            # Step 1: Search for PMIDs
            pmids = search_pubmed(term, retmax=5)
            time.sleep(rate_limit_delay)

            if not pmids:
                continue

            # Step 2: Fetch article data
            xml_data = fetch_abstracts(pmids)
            time.sleep(rate_limit_delay)

            # Step 3: Parse each article individually
            for pmid in pmids:
                # For each PMID, parse the entire XML and find its article
                parsed = parse_pubmed_article_v2(xml_data, pmid, term)
                if parsed:
                    if pmid in all_articles:
                        # Article exists, add query term if not already present
                        if term not in all_articles[pmid]["query_terms"]:
                            all_articles[pmid]["query_terms"].append(term)
                    else:
                        all_articles[pmid] = parsed

        except Exception as e:
            errors.append({"term": term, "error": str(e)})

    return list(all_articles.values()), errors, terms

In [ ]:
# Run the fixed pipeline
corpus, errors, terms = build_corpus_v2("medical_terms.csv")

print("=" * 50)
print("PART 1 - SUMMARY")
print("=" * 50)
print(f"Terms processed: {len(terms)}")
print(f"Unique articles: {len(corpus)}")
print(f"Errors: {len(errors)}")

# Save to JSON
with open("pubmed_corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, indent=2, ensure_ascii=False)

print("\n✅ Saved to pubmed_corpus.json")

# Show first article if exists
if corpus:
    print("\n📄 First article:")
    print(f"  PMID: {corpus[0]['pmid']}")
    print(f"  Title: {corpus[0]['title'][:80]}...")
    print(f"  Journal: {corpus[0]['journal']}")
    print(f"  Year: {corpus[0]['year']}")
    print(f"  Matched terms: {corpus[0]['query_terms']}")
else:
    print("\n❌ No articles were fetched. Check your internet connection.")

In [ ]:

# CORPUS CONTROL & ANALYSIS


import json
import pandas as pd
import numpy as np

# Load corpus
with open("pubmed_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print("=" * 60)
print("📚 CORPUS LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Total articles: {len(corpus)}")
print(f"Fields in each article: {list(corpus[0].keys())}")

In [ ]:
# Check each article's content quality
print("\n" + "=" * 60)
print("📋 ARTICLE CONTENT CHECK")
print("=" * 60)

for i, article in enumerate(corpus[:10], 1):
    print(f"\n{i}. PMID: {article['pmid']}")
    print(f"   Title: {article['title'][:60]}...")
    print(f"   Abstract: {article['abstract'][:80]}..." if article['abstract'] else "   Abstract: [MISSING]")
    print(f"   Journal: {article['journal']}")
    print(f"   Year: {article['year']}")
    print(f"   Terms: {article['query_terms']}")

In [ ]:
# Abstract quality analysis
print("\n" + "=" * 60)
print("📝 ABSTRACT QUALITY ANALYSIS")
print("=" * 60)

abstract_stats = {
    "total": len(corpus),
    "has_abstract": 0,
    "no_abstract": 0,
    "short_abstract": 0,  # < 50 words
    "long_abstract": 0,   # > 200 words
    "avg_length": 0
}

abstract_lengths = []

for article in corpus:
    abstract = article['abstract']
    length = len(abstract.split()) if abstract else 0
    abstract_lengths.append(length)

    if abstract and length > 0:
        abstract_stats["has_abstract"] += 1
        if length < 50:
            abstract_stats["short_abstract"] += 1
        elif length > 200:
            abstract_stats["long_abstract"] += 1
    else:
        abstract_stats["no_abstract"] += 1

abstract_stats["avg_length"] = np.mean(abstract_lengths)

print(f"\n📊 Statistics:")
print(f"   Total articles: {abstract_stats['total']}")
print(f"   With abstract: {abstract_stats['has_abstract']}")
print(f"   Without abstract: {abstract_stats['no_abstract']}")
print(f"   Short abstract (<50 words): {abstract_stats['short_abstract']}")
print(f"   Long abstract (>200 words): {abstract_stats['long_abstract']}")
print(f"   Average abstract length: {abstract_stats['avg_length']:.1f} words")
print(f"   Min abstract length: {min(abstract_lengths)} words")
print(f"   Max abstract length: {max(abstract_lengths)} words")

In [ ]:

# FIXED: Fetch ONLY articles WITH abstracts
# Each term gets 5 articles that have real abstracts

def build_corpus_abstract_only(csv_path="medical_terms.csv", target_per_term=5, max_attempts=15):
    """
    Fetch articles but ONLY include those with meaningful abstracts.
    Keeps fetching until we get 'target_per_term' articles with abstracts.
    """
    df = pd.read_csv(csv_path)
    terms = df["term"].tolist()

    all_articles = {}
    errors = []
    rate_limit_delay = 1.0 / 3

    for term in tqdm(terms, desc="Processing terms"):
        collected = 0
        attempt = 0
        offset = 0

        while collected < target_per_term and attempt < max_attempts:
            try:
                # Search with offset to get different articles
                params = {
                    "db": "pubmed",
                    "term": term,
                    "retmax": 10,
                    "retstart": offset,
                    "sort": "relevance",
                    "retmode": "json"
                }
                response = requests.get(BASE_SEARCH_URL, params=params)
                response.raise_for_status()
                data = response.json()
                pmids = data["esearchresult"]["idlist"]

                if not pmids:
                    break

                time.sleep(rate_limit_delay)

                # Fetch abstracts
                xml_data = fetch_abstracts(pmids)
                time.sleep(rate_limit_delay)

                # Check each article for abstract
                for pmid in pmids:
                    if collected >= target_per_term:
                        break

                    parsed = parse_pubmed_article_v2(xml_data, pmid, term)

                    # Check if article has meaningful abstract (>30 words)
                    if parsed and parsed['abstract']:
                        abstract_word_count = len(parsed['abstract'].split())
                        if abstract_word_count > 30:  # Meaningful abstract
                            collected += 1

                            if pmid in all_articles:
                                if term not in all_articles[pmid]["query_terms"]:
                                    all_articles[pmid]["query_terms"].append(term)
                            else:
                                all_articles[pmid] = parsed

                            print(f"   ✓ {term}: Found article {collected}/{target_per_term} (PMID: {pmid}, abstract: {abstract_word_count} words)")

                offset += 10
                attempt += 1

            except Exception as e:
                errors.append({"term": term, "error": str(e)})
                break

        if collected < target_per_term:
            errors.append({"term": term, "error": f"Only found {collected}/{target_per_term} articles with abstracts"})

    return list(all_articles.values()), errors, terms

# Run the improved pipeline
print("=" * 70)
print("🔄 FETCHING ARTICLES WITH ABSTRACTS ONLY")
print("=" * 70)

corpus_final, errors_final, terms_final = build_corpus_abstract_only("medical_terms.csv", target_per_term=5, max_attempts=15)

print("\n" + "=" * 50)
print("PART 1 - FINAL SUMMARY (Abstract Only)")
print("=" * 50)
print(f"Terms processed: {len(terms_final)}")
print(f"Unique articles (with abstracts): {len(corpus_final)}")
print(f"Errors: {len(errors_final)}")

if errors_final:
    print("\n⚠️ Errors:")
    for err in errors_final:
        print(f"   - {err['term']}: {err['error']}")

# Save to JSON
with open("pubmed_corpus_final.json", "w", encoding="utf-8") as f:
    json.dump(corpus_final, f, indent=2, ensure_ascii=False)

print("\n✅ Saved to pubmed_corpus_final.json")

# Check abstract quality
print("\n📊 ABSTRACT QUALITY CHECK:")
total_words = 0
for article in corpus_final:
    total_words += len(article['abstract'].split())
avg_words = total_words / len(corpus_final) if corpus_final else 0
print(f"   Total articles: {len(corpus_final)}")
print(f"   Average abstract length: {avg_words:.1f} words")
print(f"   All articles have abstracts: {all(bool(a['abstract']) for a in corpus_final)}")

In [ ]:

# DETAILED ANALYSIS: NEW CORPUS


import json
import numpy as np
import pandas as pd
from collections import Counter

# Load new corpus
with open("pubmed_corpus_final.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print("=" * 70)
print("📚 NEW CORPUS - DETAILED ANALYSIS")
print("=" * 70)

# 1. Basic stats
print("\n📊 BASIC STATISTICS:")
print(f"   Total articles: {len(corpus)}")
print(f"   Total unique terms: {len(set([t for a in corpus for t in a['query_terms']]))}")

# 2. Abstract length distribution
abstract_lengths = [len(a['abstract'].split()) for a in corpus]
title_lengths = [len(a['title'].split()) for a in corpus]

print(f"\n📏 LENGTH STATISTICS:")
print(f"   {'Metric':<25} {'Title':<15} {'Abstract':<15}")
print(f"   {'-'*25} {'-'*15} {'-'*15}")
print(f"   {'Average (words)':<25} {np.mean(title_lengths):.1f} {np.mean(abstract_lengths):.1f}")
print(f"   {'Median (words)':<25} {np.median(title_lengths):.0f} {np.median(abstract_lengths):.0f}")
print(f"   {'Min (words)':<25} {min(title_lengths)} {min(abstract_lengths)}")
print(f"   {'Max (words)':<25} {max(title_lengths)} {max(abstract_lengths)}")

In [ ]:
# 3. Title-abstract overlap analysis
print("\n" + "=" * 70)
print("🔗 TITLE-ABSTRACT OVERLAP ANALYSIS")
print("=" * 70)

overlap_stats = []

for article in corpus:
    title_words = set(article['title'].lower().split())
    abstract_words = set(article['abstract'].lower().split())

    # Calculate overlap
    common_words = title_words & abstract_words
    overlap_ratio = len(common_words) / len(title_words) if title_words else 0

    # Check if query terms appear in title/abstract
    query_terms = article['query_terms']
    terms_in_title = []
    terms_in_abstract = []

    for term in query_terms:
        term_lower = term.lower()
        # Check title
        if any(word in article['title'].lower() for word in term_lower.split()):
            terms_in_title.append(term)
        # Check abstract
        if any(word in article['abstract'].lower() for word in term_lower.split()):
            terms_in_abstract.append(term)

    overlap_stats.append({
        'pmid': article['pmid'],
        'title_len': len(title_words),
        'abstract_len': len(abstract_words),
        'overlap_ratio': overlap_ratio,
        'terms_in_title': len(terms_in_title),
        'terms_in_abstract': len(terms_in_abstract),
        'query_terms': query_terms
    })

print(f"\n📊 Title-Abstract Overlap:")
print(f"   Average overlap ratio: {np.mean([s['overlap_ratio'] for s in overlap_stats])*100:.1f}%")
print(f"   Median overlap ratio: {np.median([s['overlap_ratio'] for s in overlap_stats])*100:.1f}%")

print(f"\n📊 Query Term Distribution:")
print(f"   Terms only in title: {sum(1 for s in overlap_stats if s['terms_in_title'] > 0 and s['terms_in_abstract'] == 0)} articles")
print(f"   Terms only in abstract: {sum(1 for s in overlap_stats if s['terms_in_abstract'] > 0 and s['terms_in_title'] == 0)} articles")
print(f"   Terms in both: {sum(1 for s in overlap_stats if s['terms_in_title'] > 0 and s['terms_in_abstract'] > 0)} articles")

In [ ]:
# 4. Per-term quality analysis
print("\n" + "=" * 70)
print("📊 PER-TERM QUALITY ANALYSIS")
print("=" * 70)

term_analysis = {}

for article in corpus:
    for term in article['query_terms']:
        if term not in term_analysis:
            term_analysis[term] = {
                'articles': [],
                'total_abstract_len': 0,
                'title_contains_term': 0,
                'abstract_contains_term': 0
            }

        term_analysis[term]['articles'].append(article['pmid'])
        term_analysis[term]['total_abstract_len'] += len(article['abstract'].split())

        # Check if term appears in title
        term_words = term.lower().split()
        if any(word in article['title'].lower() for word in term_words):
            term_analysis[term]['title_contains_term'] += 1

        # Check if term appears in abstract
        if any(word in article['abstract'].lower() for word in term_words):
            term_analysis[term]['abstract_contains_term'] += 1

print(f"\n{'Term':<35} {'Articles':<10} {'Avg Abs Len':<12} {'Term in Title':<15} {'Term in Abstract':<15}")
print(f"{'-'*35} {'-'*10} {'-'*12} {'-'*15} {'-'*15}")

for term, data in term_analysis.items():
    avg_abs_len = data['total_abstract_len'] / len(data['articles'])
    print(f"{term:<35} {len(data['articles']):<10} {avg_abs_len:.0f} words {data['title_contains_term']}/5 {data['abstract_contains_term']}/5")

In [ ]:
# 5. Retrieval method suitability analysis
print("\n" + "=" * 70)
print("🎯 RETRIEVAL METHOD SUITABILITY ANALYSIS")
print("=" * 70)

# Calculate scores for each method
scores = {
    'BM25': 0,
    'Semantic Search': 0,
    'Hybrid RRF': 0
}

# BM25 suitability: Based on term overlap and document length
avg_overlap = np.mean([s['overlap_ratio'] for s in overlap_stats])
avg_doc_len = np.mean(abstract_lengths)

if avg_overlap > 0.3:
    scores['BM25'] += 40  # Good term overlap
elif avg_overlap > 0.2:
    scores['BM25'] += 30
else:
    scores['BM25'] += 20

if 100 < avg_doc_len < 300:
    scores['BM25'] += 30  # Ideal document length
else:
    scores['BM25'] += 20

# Semantic Search suitability: Abstract quality and multilingual
avg_abs_quality = np.mean(abstract_lengths)
if avg_abs_quality > 150:
    scores['Semantic Search'] += 40  # Rich abstracts
elif avg_abs_quality > 100:
    scores['Semantic Search'] += 35
else:
    scores['Semantic Search'] += 25

# Multilingual capability (for Turkish queries)
scores['Semantic Search'] += 30  # multilingual-e5-small supports Turkish

# Hybrid RRF suitability: Combination potential
if scores['BM25'] > 60 and scores['Semantic Search'] > 60:
    scores['Hybrid RRF'] = 90  # Both methods strong
elif scores['BM25'] > 60 or scores['Semantic Search'] > 60:
    scores['Hybrid RRF'] = 70  # One method strong
else:
    scores['Hybrid RRF'] = 50

print(f"\n📊 Method Suitability Scores (0-100):")
print(f"   🎯 BM25:            {scores['BM25']}/100")
print(f"   🧠 Semantic Search: {scores['Semantic Search']}/100")
print(f"   🔄 Hybrid RRF:      {scores['Hybrid RRF']}/100")

print(f"\n🏆 RECOMMENDATION:")
if scores['Semantic Search'] >= scores['BM25'] and scores['Semantic Search'] >= scores['Hybrid RRF']:
    print("   → SEMANTIC SEARCH is best for this corpus")
    print("   → Rich abstracts + multilingual support = better retrieval")
elif scores['BM25'] >= scores['Semantic Search'] and scores['BM25'] >= scores['Hybrid RRF']:
    print("   → BM25 is best for this corpus")
    print("   → Good term overlap and ideal document length")
else:
    print("   → HYBRID RRF is best for this corpus")
    print("   → Combines strengths of both methods")

In [ ]:
# 6. Summary table
print("\n" + "=" * 70)
print("📊 FINAL SUMMARY TABLE")
print("=" * 70)

summary_data = {
    'Metric': [
        'Total articles',
        'Average abstract length',
        'Title-Abstract overlap',
        'Terms in title (avg)',
        'Terms in abstract (avg)',
        'BM25 suitability',
        'Semantic Search suitability',
        'Hybrid RRF suitability'
    ],
    'Value': [
        f"{len(corpus)}",
        f"{np.mean(abstract_lengths):.0f} words",
        f"{np.mean([s['overlap_ratio'] for s in overlap_stats])*100:.1f}%",
        f"{np.mean([s['terms_in_title'] for s in overlap_stats]):.2f}/5",
        f"{np.mean([s['terms_in_abstract'] for s in overlap_stats]):.2f}/5",
        f"{scores['BM25']}/100",
        f"{scores['Semantic Search']}/100",
        f"{scores['Hybrid RRF']}/100"
    ]
}

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print("\n" + "=" * 70)
print("✅ ANALYSIS COMPLETE")
print("=" * 70)


# PART 2 - RETRIEVAL SYSTEM  
# Purpose: BM25, Semantic Search, Hybrid RRF


In [ ]:
import json

# Load the saved corpus
with open("pubmed_corpus_final.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"✅ Loaded {len(corpus)} articles")

# Prepare documents for retrieval
documents = []
doc_ids = []
doc_metadata = []

for article in corpus:
    # Combine title and abstract for search
    text = f"{article['title']} {article['abstract']}"
    documents.append(text)
    doc_ids.append(article['pmid'])
    doc_metadata.append({
        "pmid": article['pmid'],
        "title": article['title'],
        "abstract": article['abstract'],
        "journal": article['journal'],
        "year": article['year'],
        "query_terms": article['query_terms']
    })

print(f"✅ Prepared {len(documents)} documents")
print(f"\n📄 Sample document:")
print(f"   PMID: {doc_ids[0]}")
print(f"   Title: {doc_metadata[0]['title'][:80]}...")

In [ ]:
# Quick view of corpus structure
print("\n📋 Corpus structure:")
print(f"   Total articles: {len(corpus)}")
print(f"   Fields in each article: {list(corpus[0].keys())}")

# Show distribution of query terms
from collections import Counter
all_terms = []
for article in corpus:
    all_terms.extend(article['query_terms'])
term_counts = Counter(all_terms)

print(f"\n📊 Query term distribution:")
for term, count in term_counts.most_common():
    print(f"   {term}: {count} articles")


# PART 2A - BM25 SEARCH


In [ ]:
!pip install rank-bm25

In [ ]:

# ABSTRACT LENGTH DISTRIBUTION


import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Get abstract lengths
abstract_lengths = [len(a['abstract'].split()) for a in corpus]
title_lengths = [len(a['title'].split()) for a in corpus]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Abstract length histogram
axes[0, 0].hist(abstract_lengths, bins=15, color='steelblue', edgecolor='black')
axes[0, 0].axvline(np.mean(abstract_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(abstract_lengths):.0f}')
axes[0, 0].axvline(np.median(abstract_lengths), color='green', linestyle='--', label=f'Median: {np.median(abstract_lengths):.0f}')
axes[0, 0].set_xlabel('Abstract Length (words)')
axes[0, 0].set_ylabel('Number of Articles')
axes[0, 0].set_title('Abstract Length Distribution')
axes[0, 0].legend()

# Plot 2: Box plot
axes[0, 1].boxplot(abstract_lengths, vert=True)
axes[0, 1].set_ylabel('Abstract Length (words)')
axes[0, 1].set_title('Abstract Length Box Plot')
axes[0, 1].set_xticklabels(['All Articles'])

# Plot 3: Length by term
terms_list = []
lengths_list = []
for article in corpus:
    for term in article['query_terms']:
        terms_list.append(term)
        lengths_list.append(len(article['abstract'].split()))

term_lengths = {}
for term, length in zip(terms_list, lengths_list):
    if term not in term_lengths:
        term_lengths[term] = []
    term_lengths[term].append(length)

term_names = list(term_lengths.keys())
term_means = [np.mean(term_lengths[t]) for t in term_names]
term_stds = [np.std(term_lengths[t]) for t in term_names]

axes[1, 0].barh(term_names, term_means, xerr=term_stds, color='coral', edgecolor='black')
axes[1, 0].set_xlabel('Average Abstract Length (words)')
axes[1, 0].set_title('Abstract Length by Search Term')

# Plot 4: Cumulative distribution
sorted_lengths = np.sort(abstract_lengths)
cumulative = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)
axes[1, 1].step(sorted_lengths, cumulative, where='post', color='steelblue', linewidth=2)
axes[1, 1].axhline(0.5, color='red', linestyle='--', label='Median')
axes[1, 1].axvline(np.median(abstract_lengths), color='red', linestyle='--')
axes[1, 1].set_xlabel('Abstract Length (words)')
axes[1, 1].set_ylabel('Cumulative Proportion')
axes[1, 1].set_title('Cumulative Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("=" * 60)
print("📊 ABSTRACT LENGTH STATISTICS")
print("=" * 60)
print(f"   Min: {min(abstract_lengths)} words")
print(f"   Max: {max(abstract_lengths)} words")
print(f"   Mean: {np.mean(abstract_lengths):.1f} words")
print(f"   Median: {np.median(abstract_lengths):.0f} words")
print(f"   Std: {np.std(abstract_lengths):.1f} words")
print(f"   Q1 (25%): {np.percentile(abstract_lengths, 25):.0f} words")
print(f"   Q3 (75%): {np.percentile(abstract_lengths, 75):.0f} words")

In [ ]:
# ============================================================
# SAME TOPIC ANALYSIS: k1 and b Parameter Effects
# ============================================================

from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd

print("=" * 70)
print("SAME TOPIC ANALYSIS: k1 AND b PARAMETER EFFECTS")
print("=" * 70)

# Find gestational diabetes articles (most heterogeneous topic)
gd_articles = []
for i, meta in enumerate(doc_metadata):
    if "gestational diabetes" in meta['query_terms']:
        gd_articles.append({
            'index': i,
            'pmid': meta['pmid'],
            'title': meta['title'],
            'abstract': meta['abstract'],
            'length': len(documents[i].split())
        })

# Sort by length
gd_articles.sort(key=lambda x: x['length'])

print("\n📚 GESTATIONAL DIABETES - 5 ARTICLES (Same topic):")
print("-" * 70)
for i, a in enumerate(gd_articles, 1):
    print(f"   {i}. PMID {a['pmid']}: {a['length']:3d} words")
    print(f"      Title: {a['title'][:60]}...")

# Test documents
test_docs = [documents[a['index']] for a in gd_articles]
tokenized_docs = [doc.lower().split() for doc in test_docs]
query = "gestational diabetes"
tokenized_query = query.lower().split()

print("\n" + "=" * 70)
print("TEST 1: k1 VARIES (b=0.75 FIXED)")
print("=" * 70)

print(f"\n{'k1':>6} {'b':>6} {'1.Short':>10} {'2.':>8} {'3.':>8} {'4.':>8} {'5.Long':>10} {'Winner':>12}")
print("-" * 80)

for k1 in [0.5, 1.0, 1.5, 2.0, 2.5]:
    bm = BM25Okapi(tokenized_docs, k1=k1, b=0.75)
    scores = bm.get_scores(tokenized_query)

    winner_idx = np.argmax(scores)
    winner_len = gd_articles[winner_idx]['length']

    print(f"{k1:>6} {0.75:>6} {scores[0]:>10.4f} {scores[1]:>8.4f} {scores[2]:>8.4f} {scores[3]:>8.4f} {scores[4]:>10.4f} {winner_len:>4} words")

print("\n" + "=" * 70)
print("TEST 2: b VARIES (k1=1.5 FIXED)")
print("=" * 70)

print(f"\n{'k1':>6} {'b':>6} {'1.Short':>10} {'2.':>8} {'3.':>8} {'4.':>8} {'5.Long':>10} {'Winner':>12}")
print("-" * 80)

for b in [0.0, 0.3, 0.5, 0.75, 1.0]:
    bm = BM25Okapi(tokenized_docs, k1=1.5, b=b)
    scores = bm.get_scores(tokenized_query)

    winner_idx = np.argmax(scores)
    winner_len = gd_articles[winner_idx]['length']

    print(f"{1.5:>6} {b:>6} {scores[0]:>10.4f} {scores[1]:>8.4f} {scores[2]:>8.4f} {scores[3]:>8.4f} {scores[4]:>10.4f} {winner_len:>4} words")

print("\n" + "=" * 70)
print("TEST 3: ALL COMBINATIONS - WINNING DOCUMENT LENGTH")
print("=" * 70)

print(f"\n{'k1\\b':>6}", end="")
for b in [0.0, 0.3, 0.5, 0.75, 1.0]:
    print(f"{b:>8}", end="")
print()
print("-" * 50)

for k1 in [0.5, 1.0, 1.5, 2.0, 2.5]:
    print(f"{k1:>6}", end="")
    for b in [0.0, 0.3, 0.5, 0.75, 1.0]:
        bm = BM25Okapi(tokenized_docs, k1=k1, b=b)
        scores = bm.get_scores(tokenized_query)
        winner_idx = np.argmax(scores)
        winner_len = gd_articles[winner_idx]['length']
        print(f"{winner_len:>8}", end="")
    print()

print("\n" + "=" * 70)
print("📊 RESULTS ANALYSIS")
print("=" * 70)

# Length information
lengths = [a['length'] for a in gd_articles]
print(f"\n📏 Article lengths: {lengths}")
print(f"   Short: {lengths[0]} words, Long: {lengths[-1]} words")



# Final decision
print("=" * 70)
print("🏆 FINAL DECISION")
print("=" * 70)
print("""
   k1 = 1.5  (Default - sufficient for score magnitude)
   b  = 0.75 (Default - balanced short/long document trade-off)

   These values follow literature standards and provide
   balanced results within the same topic.
""")




# BM25 Parameter Analysis on Gestational Diabetes Corpus

## Dataset Description
- **Corpus size:** 5 articles (gestational diabetes)
- **Document length range:** 68 – 417 words  
- **Length difference:** 349 words  

---

## 1. Effect of k1 Parameter (b = 0.75 fixed)

| k1 Value | Winning Document |
|----------|-----------------|
| 0.5      | 417 words (longest) |
| 1.5      | 417 words (longest) |
| 2.5      | 417 words (longest) |

**Observations:**
- k1 does **not** change the ranking outcome
- Only affects **score magnitude** (e.g., 0.58 → 1.10)
- Controls **term frequency saturation**
- In same-topic corpus, all documents contain similar terms → k1 is not decisive

---

## 2. Effect of b Parameter (k1 = 1.5 fixed)

| b Value | Winning Document |
|--------|-----------------|
| 0.0    | 417 words (longest) |
| 0.3    | 417 words (longest) |
| 0.5    | 417 words (longest) |
| 0.75   | 417 words (longest) |
| 1.0    | 417 words (longest) |

**Observations:**
- b does **not** change the ranking outcome
- Longest document has significantly higher **term frequency**
- Even maximum length normalization (b=1.0) cannot offset tf advantage
- b controls **length normalization**, but impact is insufficient here

---

## 3. Grid Search (k1 × b)

- Total combinations tested: **25**  
  - k1 ∈ [0.5, 2.5]  
  - b ∈ [0.0, 1.0]

**Result:**
- All combinations produced the **same winner**  
- → 417-word document ranked highest in all cases  

---

## Conclusion

### 1. k1 Finding
- Does **not** affect ranking in same-topic retrieval
- Only influences score scaling
- Not decisive for document selection

### 2. b Finding
- Does **not** change ranking in this experiment
- Term frequency dominance outweighs length normalization

### 3. Why Follow Literature Defaults?

**a) No empirical difference**
- Grid search produced identical rankings
- Small corpus (5 documents) limits observable effects

**b) Literature standard**
- Recommended values from *Robertson & Zaragoza (2009)*:
  - k1 = 1.5  
  - b = 0.75  

**c) Generalization**
- Defaults are robust across domains
- More suitable for future, larger datasets

---

## Limitations of BM25

- Cannot capture **semantic meaning**
- Bias toward **longer documents** in same-topic corpora
- Cannot distinguish:
  - "diabetes in children" vs "diabetes in elderly"
- Fails on **cross-lingual queries** (e.g., Turkish queries → zero scores)

---

## Proposed Solution

Use a **Hybrid Retrieval Approach**:

- BM25 (lexical matching)
- + Semantic Search (multilingual embeddings)
- Combined using **Reciprocal Rank Fusion (RRF)**

---

## Final Decision

- **k1 = 1.5** → Standard for medical/scientific text  
- **b = 0.75** → Balanced length normalization  

**Justification:**  
No empirical difference observed. Adopted widely accepted defaults from literature. Robertson & Zaragoza (2009)

In [ ]:

# BM25 model consistently

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs, k1=1.5, b=0.75)

print("=" * 70)
print("✅ BM25 MODEL (SINGLE INSTANCE): k1=1.5, b=0.75")
print("=" * 70)


In [ ]:
# English query - Use case from assignment
english_query = "What are the latest guidelines for managing type 2 diabetes?"
tokenized_english = english_query.lower().split()
scores_english = bm25.get_scores(tokenized_english)
top_indices_english = np.argsort(scores_english)[::-1][:5]

print(f"\n🔍 Query: {english_query}")
print("\n📄 Top 5 results:")
for i, idx in enumerate(top_indices_english):
    print(f"\n   {i+1}. PMID: {doc_ids[idx]}")
    print(f"      Score: {scores_english[idx]:.4f}")
    print(f"      Title: {doc_metadata[idx]['title'][:80]}...")
    print(f"      Matched terms: {doc_metadata[idx]['query_terms']}")


In [ ]:
# Test BM25 with a sample query
sample_query = "What are the latest guidelines for managing type 2 diabetes?"

# Tokenize query
tokenized_query = sample_query.lower().split()

# Get scores
scores = bm25.get_scores(tokenized_query)

# Get top 5 results
top_indices = np.argsort(scores)[::-1][:5]

print(f"\n🔍 Query: {sample_query}")
print("\n📄 Top 5 results:")
for i, idx in enumerate(top_indices):
    print(f"\n   {i+1}. PMID: {doc_ids[idx]}")
    print(f"      Score: {scores[idx]:.4f}")
    print(f"      Title: {doc_metadata[idx]['title'][:100]}...")
    print(f"      Matched terms: {doc_metadata[idx]['query_terms']}")

In [ ]:
# Test BM25 with Turkish query
turkish_query = "Çocuklarda akut otitis media tedavisi nasıl yapılır?"

# Tokenize query (Türkçe kelimeler)
tokenized_turkish = turkish_query.lower().split()

# Get scores
scores_turkish = bm25.get_scores(tokenized_turkish)

# Get top 5 results
top_indices_turkish = np.argsort(scores_turkish)[::-1][:5]

print(f"\n🔍 Turkish Query: {turkish_query}")
print("\n📄 Top 5 results (BM25 - may not work well):")
for i, idx in enumerate(top_indices_turkish):
    print(f"\n   {i+1}. PMID: {doc_ids[idx]}")
    print(f"      Score: {scores_turkish[idx]:.4f}")
    print(f"      Title: {doc_metadata[idx]['title'][:100]}...")
    print(f"      Matched terms: {doc_metadata[idx]['query_terms']}")

In [ ]:
# Another Turkish query - harder for BM25
turkish_query2 = "Çölyak hastalığı tanı kriterleri nelerdir?"

tokenized_turkish2 = turkish_query2.lower().split()
scores_turkish2 = bm25.get_scores(tokenized_turkish2)
top_indices_turkish2 = np.argsort(scores_turkish2)[::-1][:5]

print(f"\n🔍 Turkish Query: {turkish_query2}")
print("\n📄 Top 5 results (BM25):")
for i, idx in enumerate(top_indices_turkish2):
    print(f"\n   {i+1}. PMID: {doc_ids[idx]}")
    print(f"      Score: {scores_turkish2[idx]:.4f}")
    print(f"      Title: {doc_metadata[idx]['title'][:100]}...")


# PART 2B -SEMANTIC SEARCH


In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load multilingual model (supports Turkish + English)
print("Loading multilingual embedding model...")
model = SentenceTransformer('intfloat/multilingual-e5-small')
print("✅ Model loaded")

# Create embeddings for all documents
print("Creating document embeddings...")
doc_embeddings = model.encode(documents, show_progress_bar=True)
print(f"✅ Created {len(doc_embeddings)} embeddings with dimension {doc_embeddings.shape[1]}")

In [ ]:
def semantic_search(query, top_k=5):
    """Search using semantic similarity"""
    query_embedding = model.encode([query])
    similarities = np.dot(doc_embeddings, query_embedding.T).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities

# Test with Turkish query
turkish_query = "Çölyak hastalığı tanı kriterleri nelerdir?"
indices, scores = semantic_search(turkish_query)

print(f"🔍 Semantic Search Query: {turkish_query}")
print("\n📄 Top 5 results:")
for i, idx in enumerate(indices):
    print(f"\n   {i+1}. PMID: {doc_ids[idx]}")
    print(f"      Similarity: {scores[idx]:.4f}")
    print(f"      Title: {doc_metadata[idx]['title'][:100]}...")
    print(f"      Matched terms: {doc_metadata[idx]['query_terms']}")

In [ ]:
# Test with English query
english_query = "What are the latest guidelines for managing type 2 diabetes?"
indices_en, scores_en = semantic_search(english_query)

print(f"\n🔍 Semantic Search Query: {english_query}")
print("\n📄 Top 5 results:")
for i, idx in enumerate(indices_en):
    print(f"\n   {i+1}. PMID: {doc_ids[idx]}")
    print(f"      Similarity: {scores_en[idx]:.4f}")
    print(f"      Title: {doc_metadata[idx]['title'][:100]}...")

In [ ]:
#  MedCPT
!pip install huggingface_hub -q
from huggingface_hub import list_models

results = list(list_models(search="MedCPT", limit=10))
for m in results:
    print(m.id)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np


query_tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
query_model     = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder")

article_tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
article_model     = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder")

query_model.eval()
article_model.eval()

print("✅ MedCPT yüklendi")
print(f"   Query encoder  : {sum(p.numel() for p in query_model.parameters()):,} params")
print(f"   Article encoder: {sum(p.numel() for p in article_model.parameters()):,} params")

In [ ]:
from sentence_transformers import SentenceTransformer
print("bge-m3 indiriliyor (~2.3GB, bekle)...")
bge = SentenceTransformer("BAAI/bge-m3")
print("✅ bge-m3 hazır")

In [ ]:
# ============================================================
# MODEL COMPARISON: MedCPT vs e5-small vs bge-m3
# ============================================================

import time
import numpy as np
import psutil
import os

BENCH_QUERIES = [
    ("What are the latest guidelines for managing type 2 diabetes?", "type 2 diabetes mellitus"),
    ("Çocuklarda akut otitis media tedavisi nasıl yapılır?",         "acute otitis media"),
    ("Iron supplementation dosing for anemia during pregnancy",       "iron deficiency anemia"),
    ("Çölyak hastalığı tanı kriterleri nelerdir?",                   "celiac disease diagnosis"),
    ("Antibiotic resistance patterns in community acquired pneumonia","community acquired pneumonia"),
]

def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def benchmark(name, embs, encode_q_fn):
    hit1 = hit3 = hit5 = mrr_sum = 0
    query_times = []
    for query, expected in BENCH_QUERIES:
        rel = {m["pmid"] for m in doc_metadata if expected in m["query_terms"]}
        t0  = time.time()
        q   = encode_q_fn(query)
        query_times.append(time.time() - t0)
        top5 = [doc_ids[i] for i in np.argsort(embs @ q)[::-1][:5]]
        hit1 += int(any(p in rel for p in top5[:1]))
        hit3 += int(any(p in rel for p in top5[:3]))
        hit5 += int(any(p in rel for p in top5[:5]))
        for rank, p in enumerate(top5, 1):
            if p in rel:
                mrr_sum += 1.0 / rank
                break
    n = len(BENCH_QUERIES)
    return {
        "name":     name,
        "query_ms": round(np.mean(query_times) * 1000, 1),
        "hit@1":    round(hit1/n, 2),
        "hit@3":    round(hit3/n, 2),
        "hit@5":    round(hit5/n, 2),
        "mrr":      round(mrr_sum/n, 3),
    }

# ── MedCPT embeddings
print("MedCPT encoding...")
t0 = time.time()
ram0 = get_ram_mb()
medcpt_embs = encode_articles(documents)
medcpt_enc_time = round(time.time() - t0, 1)
medcpt_ram = round(get_ram_mb() - ram0)

# ── e5-small embeddings
print("e5-small encoding...")
t0 = time.time()
ram0 = get_ram_mb()
e5_embs = model.encode(
    [f"passage: {d}" for d in documents],
    normalize_embeddings=True, batch_size=32
)
e5_enc_time = round(time.time() - t0, 1)
e5_ram = round(get_ram_mb() - ram0)

# ── bge-m3 embeddings
print("bge-m3 encoding...")
t0 = time.time()
ram0 = get_ram_mb()
bge_embs = bge.encode(documents, normalize_embeddings=True, batch_size=8)
bge_enc_time = round(time.time() - t0, 1)
bge_ram = round(get_ram_mb() - ram0)

# ── Benchmark
r_medcpt = benchmark(
    "MedCPT (ncbi)",
    medcpt_embs,
    lambda q: encode_query(q)
)
r_e5 = benchmark(
    "e5-small (multilingual)",
    e5_embs,
    lambda q: model.encode(f"query: {q}", normalize_embeddings=True)
)
r_bge = benchmark(
    "bge-m3 (BAAI)",
    bge_embs,
    lambda q: bge.encode(q, normalize_embeddings=True)
)

# Add encoding time and RAM
r_medcpt.update({"enc_s": medcpt_enc_time, "ram_mb": medcpt_ram, "dim": medcpt_embs.shape[1]})
r_e5.update({"enc_s": e5_enc_time,         "ram_mb": e5_ram,     "dim": e5_embs.shape[1]})
r_bge.update({"enc_s": bge_enc_time,       "ram_mb": bge_ram,    "dim": bge_embs.shape[1]})

# ── Summary table
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(f"{'Model':<26} {'Enc(s)':>6} {'RAM MB':>7} {'Dim':>5} {'Q ms':>6} {'H@1':>5} {'H@3':>5} {'H@5':>5} {'MRR':>6}")
print("-" * 80)
for r in [r_medcpt, r_e5, r_bge]:
    print(f"{r['name']:<26} {r['enc_s']:>6} {r['ram_mb']:>7} "
          f"{r['dim']:>5} {r['query_ms']:>6} "
          f"{r['hit@1']:>5} {r['hit@3']:>5} {r['hit@5']:>5} {r['mrr']:>6}")

best_mrr   = max([r_medcpt, r_e5, r_bge], key=lambda x: x["mrr"])
best_speed = min([r_medcpt, r_e5, r_bge], key=lambda x: x["query_ms"])
best_bal   = max([r_medcpt, r_e5, r_bge], key=lambda x: x["mrr"] / (x["ram_mb"] + 1))

print(f"\n  Best MRR     : {best_mrr['name']}")
print(f"  Fastest      : {best_speed['name']}")
print(f"  Best balance : {best_bal['name']}")

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)



# PART 2C -Hybrid — Reciprocal Rank Fusion


In [ ]:
# ============================================================
# QUERY LANGUAGE ANALYSIS — TR vs EN per model
# ============================================================

import numpy as np

# Turkish test queries
TR_QUERIES = [
    ("Çocuklarda akut otitis media tedavisi nasıl yapılır?", "acute otitis media"),
    ("Çölyak hastalığı tanı kriterleri nelerdir?",           "celiac disease diagnosis"),
]

# English test queries
EN_QUERIES = [
    ("What are the latest guidelines for managing type 2 diabetes?", "type 2 diabetes mellitus"),
    ("Iron supplementation dosing for anemia during pregnancy",       "iron deficiency anemia"),
    ("Antibiotic resistance patterns in community acquired pneumonia","community acquired pneumonia"),
]

def mrr_score(embs, encode_q_fn, queries):
    """Calculate Mean Reciprocal Rank for a set of queries"""
    total = 0
    for query, expected in queries:
        rel = {m["pmid"] for m in doc_metadata if expected in m["query_terms"]}
        q = encode_q_fn(query)
        top5 = [doc_ids[i] for i in np.argsort(embs @ q)[::-1][:5]]
        for rank, p in enumerate(top5, 1):
            if p in rel:
                total += 1.0 / rank
                break
    return round(total / len(queries), 3)

# Models to compare
models = [
    ("MedCPT",    medcpt_embs, lambda q: encode_query(q)),
    ("e5-small",  e5_embs,     lambda q: model.encode(f"query: {q}", normalize_embeddings=True)),
    ("bge-m3",    bge_embs,    lambda q: bge.encode(q, normalize_embeddings=True)),
]

print("=" * 55)
print("MRR — TR vs EN per model")
print("=" * 55)
print(f"{'Model':<12} {'TR MRR':>8} {'EN MRR':>8} {'Gap':>8}")
print("-" * 40)

results_lang = []
for name, embs, enc_fn in models:
    tr_mrr = mrr_score(embs, enc_fn, TR_QUERIES)
    en_mrr = mrr_score(embs, enc_fn, EN_QUERIES)
    gap = round(en_mrr - tr_mrr, 3)
    results_lang.append({"name": name, "tr": tr_mrr, "en": en_mrr, "gap": gap})
    print(f"{name:<12} {tr_mrr:>8} {en_mrr:>8} {gap:>8}")

print()
print("Gap = EN - TR (positive gap indicates weaker Turkish performance)")

print("\n" + "=" * 55)
print("INTERPRETATION")
print("=" * 55)


In [ ]:

# HYBRID RRF — Which combination can match the bge-m3?


def mrr_hybrid(sem_embs, encode_q_fn, queries, k_rrf=60):
    total = 0
    for query, expected in queries:
        rel = {m["pmid"] for m in doc_metadata if expected in m["query_terms"]}

        # BM25
        bm25_sc  = bm25.get_scores(query.lower().split())
        bm25_rnk = list(np.argsort(bm25_sc)[::-1])

        # Semantic
        q_emb   = encode_q_fn(query)
        sem_sc  = sem_embs @ q_emb
        sem_rnk = list(np.argsort(sem_sc)[::-1])

        # RRF
        rrf  = reciprocal_rank_fusion({"bm25": bm25_rnk, "semantic": sem_rnk}, k=k_rrf)
        top5 = [doc_ids[i] for i, _ in rrf[:5]]

        for rank, p in enumerate(top5, 1):
            if p in rel:
                total += 1.0 / rank
                break
    return round(total / len(queries), 3)

ALL_QUERIES = TR_QUERIES + EN_QUERIES

combos = [
    ("BM25 only",          None,        None),
    ("MedCPT only",        medcpt_embs, lambda q: encode_query(q)),
    ("e5-small only",      e5_embs,     lambda q: model.encode(f"query: {q}", normalize_embeddings=True)),
    ("bge-m3 only",        bge_embs,    lambda q: bge.encode(q, normalize_embeddings=True)),
    ("BM25 + MedCPT RRF",  medcpt_embs, lambda q: encode_query(q)),
    ("BM25 + e5 RRF",      e5_embs,     lambda q: model.encode(f"query: {q}", normalize_embeddings=True)),
    ("BM25 + bge-m3 RRF",  bge_embs,    lambda q: bge.encode(q, normalize_embeddings=True)),
]

print("=" * 65)
print("FULL COMPARISON — Semantic only vs Hybrid RRF")
print("=" * 65)
print(f"{'Combo':<22} {'TR MRR':>8} {'EN MRR':>8} {'ALL MRR':>9} {'RAM':>8}")
print("-" * 60)

ram_map = {"MedCPT": "100MB", "e5-small": "203MB", "bge-m3": "1473MB", "BM25": "~0MB"}

for name, embs, enc_fn in combos:
    if embs is None:
        # BM25 only
        def bm25_only_mrr(queries):
            total = 0
            for query, expected in queries:
                rel  = {m["pmid"] for m in doc_metadata if expected in m["query_terms"]}
                sc   = bm25.get_scores(query.lower().split())
                top5 = [doc_ids[i] for i in np.argsort(sc)[::-1][:5]]
                for rank, p in enumerate(top5, 1):
                    if p in rel:
                        total += 1.0 / rank
                        break
            return round(total / len(queries), 3)
        tr_  = bm25_only_mrr(TR_QUERIES)
        en_  = bm25_only_mrr(EN_QUERIES)
        all_ = bm25_only_mrr(ALL_QUERIES)
        ram  = "~0MB"
    elif "RRF" in name:
        tr_  = mrr_hybrid(embs, enc_fn, TR_QUERIES)
        en_  = mrr_hybrid(embs, enc_fn, EN_QUERIES)
        all_ = mrr_hybrid(embs, enc_fn, ALL_QUERIES)
        ram  = [v for k, v in ram_map.items() if k in name][0]
    else:
        tr_  = mrr_score(embs, enc_fn, TR_QUERIES)
        en_  = mrr_score(embs, enc_fn, EN_QUERIES)
        all_ = mrr_score(embs, enc_fn, ALL_QUERIES)
        model_key = name.split()[0]
        ram  = ram_map.get(model_key, "?")

    marker = " ← bge-m3 baseline" if name == "bge-m3 only" else ""
    print(f"{name:<22} {tr_:>8} {en_:>8} {all_:>9} {ram:>8}{marker}")

print()


In [ ]:
# e5_embs set et — boyut 384
doc_embeddings = e5_embs
print(f"✅ doc_embeddings = e5_embs | dim={doc_embeddings.shape[1]}")

In [ ]:
# ============================================================
# RECIPROCAL RANK FUSION (RRF)
# Cormack, Clarke & Butt (2009) SIGIR
# ============================================================

import numpy as np

def reciprocal_rank_fusion(rankings_dict: dict, k: int = 60) -> list:
    """
    RRF(d) = Σ 1 / (k + rank(d))

    Args:
        rankings_dict: {"bm25": [idx0, idx1, ...], "semantic": [...]}
                       Each list contains document indices, rank starts from 1
        k: Smoothing constant, default=60 (optimal from paper)
    Returns:
        List of [(doc_idx, rrf_score)] sorted descending
    """
    scores = {}
    for method, ranked_indices in rankings_dict.items():
        for rank, doc_idx in enumerate(ranked_indices, start=1):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def run_hybrid(query, top_k=5, k_rrf=60):
    """
    Hybrid search combining BM25 and Semantic Search using RRF
    """
    # BM25 scoring
    bm25_sc = bm25.get_scores(query.lower().split())
    bm25_ranked = list(np.argsort(bm25_sc)[::-1])

    # Semantic search with e5-small (query prefix required)
    q_emb = model.encode(f"query: {query}", normalize_embeddings=True)
    sem_sc = doc_embeddings @ q_emb
    sem_ranked = list(np.argsort(sem_sc)[::-1])

    # RRF fusion
    rrf = reciprocal_rank_fusion(
        {"bm25": bm25_ranked, "semantic": sem_ranked}, k=k_rrf
    )

    return {
        "bm25": [(doc_ids[i], bm25_sc[i]) for i in bm25_ranked[:top_k]],
        "semantic": [(doc_ids[i], sem_sc[i]) for i in sem_ranked[:top_k]],
        "hybrid": [(doc_ids[i], s) for i, s in rrf[:top_k]],
    }

# ============================================================
# EFFECT OF k PARAMETER
# ============================================================

print("=" * 60)
print("EFFECT OF k PARAMETER ON RRF")
print("=" * 60)
print("k=0    → 1/(0+1)=1.0   extreme weight on top1, very sharp")
print("k=60   → 1/61≈0.016    balanced, paper recommendation")
print("k=1000 → 1/1001≈0.001  all ranks almost equal, no discrimination")
print()

q = "What are the latest guidelines for managing type 2 diabetes?"
bm25_sc = bm25.get_scores(q.lower().split())
q_emb = model.encode(f"query: {q}", normalize_embeddings=True)
sem_sc = doc_embeddings @ q_emb
bm25_rnk = list(np.argsort(bm25_sc)[::-1])
sem_rnk = list(np.argsort(sem_sc)[::-1])

print(f"{'k':>6} {'1st-2nd gap':>12}  Top3")
print("-" * 65)
for k_val in [0, 1, 10, 60, 1000]:
    rrf = reciprocal_rank_fusion({"bm25": bm25_rnk, "semantic": sem_rnk}, k=k_val)
    gap = rrf[0][1] - rrf[1][1]
    top3 = [doc_metadata[i]["title"][:30] + "..." for i, _ in rrf[:3]]
    print(f"{k_val:>6} {gap:>12.6f}  {top3[0]}")
    print(f"{'':>20}  {top3[1]}")
    print(f"{'':>20}  {top3[2]}")
    print()

# ============================================================
# WHY USE RANK POSITION INSTEAD OF RAW SCORES?
# ============================================================

print("=" * 60)
print("WHY RANK POSITION INSTEAD OF RAW SCORES?")
print("=" * 60)
sample_q = "type 2 diabetes treatment"
bm25_sc2 = bm25.get_scores(sample_q.lower().split())
q_emb2 = model.encode(f"query: {sample_q}", normalize_embeddings=True)
sem_sc2 = doc_embeddings @ q_emb2
top_idx = np.argsort(bm25_sc2)[::-1][0]

print(f"Example — For the top-1 article:")
print(f"  BM25 score  : {bm25_sc2[top_idx]:.4f}  (TF/IDF based, unbounded scale)")
print(f"  Cosine sim  : {sem_sc2[top_idx]:.4f}  (normalized, 0-1 range)")
print(f"  Direct sum  : {bm25_sc2[top_idx] + sem_sc2[top_idx]:.4f}  ← BM25 dominates!")
print(f"  Rank pos    : Both are rank 1 → RRF score is high")
print()
print("BM25 scores range ~8-15, cosine similarity ~0.8-0.9.")
print("If summed directly, BM25 always dominates — cosine becomes meaningless.")
print("Rank position normalizes the scale and removes bias.")

# ============================================================
# 5-QUERY EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("5-QUERY EVALUATION — BM25 vs SEMANTIC vs HYBRID")
print("=" * 60)

EVAL_QUERIES = [
    ("What are the latest guidelines for managing type 2 diabetes?", "type 2 diabetes mellitus"),
    ("Çocuklarda akut otitis media tedavisi nasıl yapılır?",         "acute otitis media"),
    ("Iron supplementation dosing for anemia during pregnancy",       "iron deficiency anemia"),
    ("Çölyak hastalığı tanı kriterleri nelerdir?",                   "celiac disease diagnosis"),
    ("Antibiotic resistance patterns in community acquired pneumonia","community acquired pneumonia"),
]

for query, expected in EVAL_QUERIES:
    rel = {m["pmid"] for m in doc_metadata if expected in m["query_terms"]}
    result = run_hybrid(query)
    print(f"\nQ: {query[:58]}...")
    for method in ["bm25", "semantic", "hybrid"]:
        top1_pmid = result[method][0][0]
        hit = "✓" if top1_pmid in rel else "✗"
        top1_title = next(m["title"][:38] for m in doc_metadata
                          if m["pmid"] == top1_pmid)
        print(f"  {method:>9}: [{hit}] {top1_title}...")

# Set document embeddings
doc_embeddings = e5_embs
print("\n✅ RRF ready | doc_embeddings = e5_embs | k=60")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)


# ============================================
# PART 2D -Evaluate
# ============================================

In [ ]:

# PART 2D — EVALUATION: MRR + Hit@1 + Hit@3 + Hit@5


import numpy as np

QUERY_TERM_MAP = {
    "What are the latest guidelines for managing type 2 diabetes?": "type 2 diabetes mellitus",
    "Çocuklarda akut otitis media tedavisi nasıl yapılır?":         "acute otitis media",
    "Iron supplementation dosing for anemia during pregnancy":       "iron deficiency anemia",
    "Çölyak hastalığı tanı kriterleri nelerdir?":                   "celiac disease diagnosis",
    "Antibiotic resistance patterns in community acquired pneumonia":"community acquired pneumonia",
}

def evaluate_all(query, expected_term):
    """Evaluate BM25, Semantic, and Hybrid RRF for a single query"""
    rel = {m["pmid"] for m in doc_metadata if expected_term in m["query_terms"]}

    # BM25 scores and ranking
    bm25_sc = bm25.get_scores(query.lower().split())
    bm25_top = [doc_ids[i] for i in np.argsort(bm25_sc)[::-1][:5]]

    # Semantic search with normalized embeddings
    q_emb = model.encode(f"query: {query}", normalize_embeddings=True)
    sem_sc = doc_embeddings @ q_emb
    sem_top = [doc_ids[i] for i in np.argsort(sem_sc)[::-1][:5]]

    # Hybrid RRF fusion (k=60 as per paper)
    rrf = reciprocal_rank_fusion(
        {"bm25": list(np.argsort(bm25_sc)[::-1]),
         "semantic": list(np.argsort(sem_sc)[::-1])}, k=60
    )
    hyb_top = [doc_ids[i] for i, _ in rrf[:5]]

    results = {}
    for method, top5 in [("bm25", bm25_top), ("semantic", sem_top), ("hybrid", hyb_top)]:
        mrr_val = 0
        for rank, p in enumerate(top5, 1):
            if p in rel:
                mrr_val = 1.0 / rank
                break
        results[method] = {
            "mrr": mrr_val,
            "hit@1": int(any(p in rel for p in top5[:1])),
            "hit@3": int(any(p in rel for p in top5[:3])),
            "hit@5": int(any(p in rel for p in top5[:5])),
            "top1": next((m["title"][:40] for m in doc_metadata if m["pmid"] == top5[0]), "?"),
            "top1_hit": top5[0] in rel,
        }
    return results

# Run evaluation on all queries
all_results = {}
agg = {m: {"mrr": [], "h1": [], "h3": [], "h5": []}
       for m in ["bm25", "semantic", "hybrid"]}

print("=" * 70)
print("PER-QUERY RESULTS")
print("=" * 70)

for query, term in QUERY_TERM_MAP.items():
    lang = "TR" if any(c in query for c in "çğışöüÇĞİŞÖÜ") else "EN"
    res = evaluate_all(query, term)
    all_results[query] = res

    print(f"\n[{lang}] {query[:55]}...")
    print(f"      Expected: '{term}'")
    print(f"      {'Method':<10} {'MRR':>6} {'H@1':>5} {'H@3':>5} {'H@5':>5}  Top1")
    print(f"      {'-'*65}")
    for method in ["bm25", "semantic", "hybrid"]:
        r = res[method]
        hit = "✓" if r["top1_hit"] else "✗"
        print(f"      {method:<10} {r['mrr']:>6.3f} {r['hit@1']:>5} {r['hit@3']:>5} {r['hit@5']:>5}  [{hit}] {r['top1']}...")
        agg[method]["mrr"].append(r["mrr"])
        agg[method]["h1"].append(r["hit@1"])
        agg[method]["h3"].append(r["hit@3"])
        agg[method]["h5"].append(r["hit@5"])

# Summary table
print("\n" + "=" * 55)
print("EVALUATION SUMMARY (5 queries)")
print("=" * 55)
print(f"{'Method':<12} {'MRR':>8} {'Hit@1':>7} {'Hit@3':>7} {'Hit@5':>7}")
print("-" * 45)
for method in ["bm25", "semantic", "hybrid"]:
    a = agg[method]
    print(f"{method:<12} {np.mean(a['mrr']):>8.3f} "
          f"{np.mean(a['h1']):>7.2f} "
          f"{np.mean(a['h3']):>7.2f} "
          f"{np.mean(a['h5']):>7.2f}")

best = max(["bm25", "semantic", "hybrid"], key=lambda m: np.mean(agg[m]["mrr"]))
print(f"\n  Best method : {best.upper()}")
print(f"  Metric used : MRR (Mean Reciprocal Rank)")
print(f"  Ground truth: pseudo-relevance via query_terms field")
print(f"  Note: TR queries confirm semantic+hybrid > BM25 alone")

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)



# PART 3 - RAG GENERATION
# Purpose: LLM-powered answer generation with citations


In [ ]:
!pip install -q google-genai

In [ ]:

from getpass import getpass
from google.colab import userdata
from google import genai
from google.genai import types
from collections import Counter

print("✅ All libraries imported successfully")

In [ ]:
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print(f"✅ API key loaded: {GEMINI_API_KEY[:10]}...")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please check secret name in Colab Secrets")

In [ ]:

# FIX: CORRECT MODEL NAMES FOR NEW GOOGLE GENAI SDK


print("=" * 70)
print("CHECKING AVAILABLE GEMINI MODELS")
print("=" * 70)

# List available models to see correct names
try:
    for model_info in client.models.list():
        if 'gemini' in model_info.name:
            print(f"   Available: {model_info.name}")
except Exception as e:
    print(f"   Could not list models: {e}")

# Try different model name formats
MODEL_CANDIDATES = [
    "gemini-2.0-flash-exp",      # Experimental
    "gemini-2.0-flash",          # Stable 2.0
    "gemini-1.5-flash",          # Old name (not working)
    "models/gemini-1.5-flash",   # With prefix
    "gemini-pro",                 # Legacy
]

print("\n" + "=" * 70)
print("TESTING MODEL NAMES")
print("=" * 70)

WORKING_MODEL = None

for model_name in MODEL_CANDIDATES:
    try:
        test_response = client.models.generate_content(
            model=model_name,
            contents="Say 'OK' if you work.",
            config=types.GenerateContentConfig(max_output_tokens=10)
        )
        print(f"✅ WORKING: {model_name}")
        WORKING_MODEL = model_name
        break
    except Exception as e:
        print(f"❌ FAILED: {model_name} - {str(e)[:50]}...")

if WORKING_MODEL is None:
    print("\n⚠️ No model found. Using fallback: gemini-2.0-flash-exp")
    WORKING_MODEL = "gemini-2.0-flash-exp"

print(f"\n📌 SELECTED MODEL: {WORKING_MODEL}")

In [ ]:
# ============================================================
# PART 3 - FINAL RAG DEMO
# Hybrid RRF + Gemini (Gemini 3.1 Flash Lite)
# ============================================================

import requests
import numpy as np
import json
import time

print("=" * 80)
print("PART 3: FINAL RAG DEMO - HYBRID RRF + GEMINI 3.1 FLASH LITE")
print("=" * 80)

# Gemini API key from Colab secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Rate limit delay (seconds between API calls to avoid 429 errors)
RATE_LIMIT_DELAY = 3.0  # 3 seconds between requests

# Using Gemini 3.1 Flash Lite (lower rate limits, more stable)
MODEL_NAME = "gemini-3.1-flash-lite-preview"

print(f"📌 Using model: {MODEL_NAME}")
print(f"📌 Rate limit delay: {RATE_LIMIT_DELAY}s between requests")

# ============================================================
# WORKING RRF FUNCTION (from Part 2C)
# ============================================================

def run_hybrid(query, top_k=5, k_rrf=60):
    """
    Hybrid search combining BM25 and Semantic Search using RRF
    """
    # BM25 scoring
    bm25_sc = bm25.get_scores(query.lower().split())
    bm25_ranked = list(np.argsort(bm25_sc)[::-1])

    # Semantic search with query prefix and normalization
    q_emb = model.encode(f"query: {query}", normalize_embeddings=True)
    sem_sc = doc_embeddings @ q_emb
    sem_ranked = list(np.argsort(sem_sc)[::-1])

    # RRF fusion (k=60 as per Cormack et al. 2009)
    scores = {}
    for rank, doc_idx in enumerate(bm25_ranked, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k_rrf + rank)
    for rank, doc_idx in enumerate(sem_ranked, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k_rrf + rank)

    rrf = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return {
        "bm25": [(doc_ids[i], bm25_sc[i]) for i in bm25_ranked[:top_k]],
        "semantic": [(doc_ids[i], sem_sc[i]) for i in sem_ranked[:top_k]],
        "hybrid": [(doc_ids[i], s) for i, s in rrf[:top_k]],
    }

# ============================================================
# SAFE RAG ANSWER FUNCTION WITH RATE LIMITING
# ============================================================

def rag_answer_safe(query, top_k=3):
    """
    Generate cited answer using Hybrid RRF + Gemini 3.1 Flash Lite
    Includes rate limiting to avoid API quota errors
    """

    print(f"   🔍 Searching: {query[:50]}...")

    try:
        # Get hybrid search results
        result = run_hybrid(query, top_k=top_k)
        hybrid_results = result["hybrid"]

        # Extract retrieved documents
        retrieved_docs = []
        for pmid, score in hybrid_results:
            for meta in doc_metadata:
                if meta['pmid'] == pmid:
                    retrieved_docs.append(meta)
                    break

        # Build context for LLM
        context = ""
        sources = []
        for i, doc in enumerate(retrieved_docs, 1):
            context += f"\n--- DOCUMENT {i} ---\n"
            context += f"Title: {doc['title']}\n"
            context += f"Journal: {doc['journal']}, Year: {doc['year']}\n"
            context += f"Abstract: {doc['abstract'][:400]}\n"
            sources.append(f"[{i}] PMID {doc['pmid']}")

        # System prompt for medical QA
        prompt = f"""You are a medical AI assistant. Answer based ONLY on the context below. Cite sources with [1], [2]. Answer in the same language as the question.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

        # Rate limiting - wait before API call
        time.sleep(RATE_LIMIT_DELAY)

        # Call Gemini REST API with Gemini 3.1 Flash Lite
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent?key={GEMINI_API_KEY}"
        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": 0.2, "maxOutputTokens": 500}
        }

        response = requests.post(url, json=payload)

        # Parse response safely
        if response.status_code == 200:
            result_json = response.json()
            if 'candidates' in result_json and len(result_json['candidates']) > 0:
                answer = result_json['candidates'][0]['content']['parts'][0]['text']
            else:
                answer = "API returned no candidates. Please try again."
        elif response.status_code == 429:
            answer = "API rate limit exceeded. Please wait and try again later."
            print(f"   ⚠️ Rate limit hit! Waiting {RATE_LIMIT_DELAY * 2}s before next...")
            time.sleep(RATE_LIMIT_DELAY * 2)
        else:
            answer = f"API Error {response.status_code}: {response.text[:100]}"

    except Exception as e:
        answer = f"Error: {str(e)}"
        retrieved_docs = []
        sources = []

    return answer, sources, retrieved_docs

# ============================================================
# TEST 1: ORIGINAL 5 USE CASE QUERIES
# ============================================================

print("\n" + "=" * 80)
print("📋 TEST 1: USE CASE QUERIES (Original 5 from assignment)")
print("=" * 80)

use_case_queries = [
    "What are the latest guidelines for managing type 2 diabetes?",
    "Çocuklarda akut otitis media tedavisi nasıl yapılır?",
    "Iron supplementation dosing for anemia during pregnancy",
    "Çölyak hastalığı tanı kriterleri nelerdir?",
    "Antibiotic resistance patterns in community acquired pneumonia"
]

for i, query in enumerate(use_case_queries, 1):
    print("\n" + "=" * 70)
    print(f"🔍 QUERY {i}: {query}")
    print("=" * 70)

    answer, sources, docs = rag_answer_safe(query, top_k=3)

    print("\n📚 RETRIEVED DOCUMENTS:")
    for j, doc in enumerate(docs, 1):
        print(f"   [{j}] {doc['title'][:70]}...")
        print(f"       {doc['journal']} ({doc['year']})")

    print("\n💡 ANSWER:")
    print("-" * 50)
    print(answer[:800] if len(answer) > 800 else answer)
    print("-" * 50)

    if sources:
        print("\n📖 SOURCES:", ", ".join(sources))

# ============================================================
# TEST 2: TURKISH QUERIES
# ============================================================

print("\n" + "=" * 80)
print("📋 TEST 2: TURKISH QUERIES (Multilingual capability test)")
print("=" * 80)

turkish_queries = [
    "Çölyak hastalığı nasıl teşhis edilir?",
    "Çocuklarda orta kulak iltihabı tedavisi",
    "Demir eksikliği anemisi nedir?"
]

for query in turkish_queries:
    print("\n" + "=" * 70)
    print(f"🔍 QUERY: {query}")
    print("=" * 70)

    answer, sources, docs = rag_answer_safe(query, top_k=3)

    print("\n📚 RETRIEVED DOCUMENTS:")
    for j, doc in enumerate(docs, 1):
        print(f"   [{j}] {doc['title'][:60]}...")
        print(f"       {doc['journal']} ({doc['year']})")

    print("\n💡 ANSWER:")
    print("-" * 50)
    print(answer[:800] if len(answer) > 800 else answer)
    print("-" * 50)

print("\n" + "=" * 80)
print("✅ RAG DEMO COMPLETE")
print("=" * 80)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("📊 FINAL RAG SYSTEM SUMMARY")
print("=" * 80)


In [ ]:
from google.colab import files
files.download('pubmed_corpus_final.json')

In [ ]:
import json

ground_truth = {
  "ground_truth": {
    "atrial_fibrillation": [
      {
        "question": "What is atrial fibrillation?",
        "expected_answer": "clinical overview of atrial fibrillation, focusing on diagnosis, treatment, and practice improvement",
        "source_pmid": "28265666",
        "source_title": "Atrial Fibrillation."
      },
      {
        "question": "How is atrial fibrillation managed?",
        "expected_answer": "stroke prevention through systemic anticoagulation will be a cornerstone of atrial fibrillation management",
        "source_pmid": "16326228",
        "source_title": "Atrial fibrillation."
      },
      {
        "question": "What causes atrial fibrillation in young patients?",
        "expected_answer": "AF in the young is unique, with a distinct set of pathophysiologic factors that influence disease onset, response to treatment, and prognosis",
        "source_pmid": "40436475",
        "source_title": "Atrial Fibrillation in the Young: Pathogenesis and Clinical Implications."
      },
      {
        "question": "What is the neurogenic theory of atrial fibrillation?",
        "expected_answer": "autonomic imbalance is not a simple modulation factor; both the trigger and the substrate of atrial fibrillation can be influenced by abnormal cardiac innervation",
        "source_pmid": "29229215",
        "source_title": "Atrial fibrillation: Neurogenic or myogenic?"
      },
      {
        "question": "Is catheter ablation effective for AF in heart failure?",
        "expected_answer": "catheter ablation in patients with atrial fibrillation and concomitant heart failure is effective",
        "source_pmid": "30561633",
        "source_title": "Atrial fibrillation ablation in heart failure."
      }
    ],
    "type_2_diabetes_mellitus": [
      {
        "question": "What is type 2 diabetes mellitus associated with?",
        "expected_answer": "strongly associated with lower performance on multiple domains of cognitive function and with structural abnormalities of the brain",
        "source_pmid": "34251351",
        "source_title": "Type 2 Diabetes Mellitus and Cognitive Impairment."
      },
      {
        "question": "What pharmacological approaches prevent type 2 diabetes?",
        "expected_answer": "certain drugs prevent or slow development of hyperglycemia; drugs used for obesity management were shown to prevent T2DM",
        "source_pmid": "36967777",
        "source_title": "Pharmacological approaches to the prevention of type 2 diabetes mellitus."
      },
      {
        "question": "What is the role of gut microbiota in type 2 diabetes?",
        "expected_answer": "gut microbiota plays an important role in the development of metabolic diseases, especially T2DM",
        "source_pmid": "35242721",
        "source_title": "Gut Microbiota: An Important Player in Type 2 Diabetes Mellitus."
      },
      {
        "question": "What are the pathophysiological mechanisms of type 2 diabetes?",
        "expected_answer": "mitochondrial dysfunction, obesity, gut microbiota, oxidative stress and inflammation",
        "source_pmid": "40416523",
        "source_title": "Associated factors and principal pathophysiological mechanisms of type 2 diabetes mellitus."
      }
    ],
    "acute_otitis_media": [
      {
        "question": "What percentage of children get acute otitis media?",
        "expected_answer": "affects over 80% of children before their third birthday",
        "source_pmid": "28707578",
        "source_title": "Acute Otitis Media in Children."
      },
      {
        "question": "What is the first-line antibiotic for acute otitis media?",
        "expected_answer": "Amoxicillin is the drug of choice",
        "source_pmid": "28707578",
        "source_title": "Acute Otitis Media in Children."
      },
      {
        "question": "When can antibiotic therapy be deferred in AOM?",
        "expected_answer": "in children two years or older with mild symptoms",
        "source_pmid": "24134083",
        "source_title": "Otitis media: diagnosis and treatment."
      },
      {
        "question": "What are the common bacterial causes of AOM?",
        "expected_answer": "Streptococcus pneumoniae, Haemophilus influenzae, and Moraxella catarrhalis",
        "source_pmid": "24439877",
        "source_title": "Acute otitis media."
      },
      {
        "question": "What is the difference between AOM and OME?",
        "expected_answer": "the fluid is not infected in OME as is seen in AOM patients",
        "source_pmid": "24439877",
        "source_title": "Acute otitis media."
      }
    ],
    "celiac_disease_diagnosis": [
      {
        "question": "What is the gold standard for celiac disease diagnosis?",
        "expected_answer": "duodenal biopsy is the gold standard",
        "source_pmid": "26784474",
        "source_title": "Contemporary celiac disease diagnosis: is a biopsy avoidable?"
      },
      {
        "question": "What serological test is used for celiac disease?",
        "expected_answer": "IgA anti-transglutaminase (TG2) antibodies combined with IgA quantification to rule out IgA deficiency",
        "source_pmid": "33878915",
        "source_title": "Celiac disease: Understandings in diagnostic, nutritional, and medicinal aspects."
      },
      {
        "question": "What is the no-biopsy approach for celiac disease?",
        "expected_answer": "conditional no-biopsy approach for selected adults with high-titre IgA anti-TG2 serology (≥10×ULN)",
        "source_pmid": "40999951",
        "source_title": "European Society for the Study of Coeliac Disease 2025 Updated Guidelines..."
      },
      {
        "question": "How many duodenal biopsies are recommended?",
        "expected_answer": "at least four samples from the second part of the duodenum, with bulb biopsies conditionally included",
        "source_pmid": "40999951",
        "source_title": "European Society for the Study of Coeliac Disease 2025 Updated Guidelines..."
      },
      {
        "question": "What is the treatment for celiac disease?",
        "expected_answer": "primarily a gluten-free diet (GFD)",
        "source_pmid": "31210940",
        "source_title": "European Society for the Study of Coeliac Disease (ESsCD) guideline for coeliac disease and other gluten-related disorders."
      }
    ]
  },
  "metadata": {
    "created_date": "2026-04-18",
    "source": "pubmed_corpus_final.json",
    "total_questions": 24,
    "terms_covered": [
      "atrial_fibrillation",
      "type_2_diabetes_mellitus",
      "acute_otitis_media",
      "celiac_disease_diagnosis"
    ]
  }
}

# Save to file
with open("ground_truth.json", "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

print("✅ ground_truth.json saved!")

# Download
from google.colab import files
files.download("ground_truth.json")

In [ ]:
# ============================================================
# GROUND TRUTH EVALUATION - RAG Answers vs Expected Answers
# ============================================================

import json
import numpy as np
from difflib import SequenceMatcher

# ============================================================
# 1. LOAD GROUND TRUTH
# ============================================================

with open("ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

print("=" * 70)
print("📋 GROUND TRUTH LOADED")
print("=" * 70)

# Flatten ground truth into a list with indices
all_questions = []
idx = 0

for term, questions in ground_truth["ground_truth"].items():
    for q in questions:
        all_questions.append({
            "id": idx,
            "term": term,
            "question": q["question"],
            "expected_answer": q["expected_answer"],
            "source_pmid": q["source_pmid"],
            "source_title": q["source_title"]
        })
        idx += 1

print(f"Total questions in ground truth: {len(all_questions)}")
print(f"\n📊 Sample questions:")
for i in range(min(5, len(all_questions))):
    print(f"   [{all_questions[i]['id']}] {all_questions[i]['question'][:60]}...")

In [ ]:
# ============================================================
# 2. FUNCTION TO EVALUATE RAG ANSWER
# ============================================================

def evaluate_rag_answer(rag_answer, expected_answer, retrieved_pmids, expected_pmid):
    """
    Evaluate RAG answer against ground truth

    Returns:
        score: 0-100 based on multiple metrics
        details: breakdown of scores
    """
    scores = {}

    # 1. Retrieval accuracy (does it find the right document?)
    if expected_pmid in retrieved_pmids:
        scores["retrieval"] = 100
    else:
        scores["retrieval"] = 0

    # 2. Answer similarity (using fuzzy matching)
    similarity = SequenceMatcher(None,
                                  rag_answer.lower()[:200],
                                  expected_answer.lower()[:200]).ratio()
    scores["similarity"] = round(similarity * 100, 1)

    # 3. Contains key terms from expected answer
    expected_words = set(expected_answer.lower().split())
    rag_words = set(rag_answer.lower().split())
    common_words = expected_words & rag_words
    scores["key_terms"] = round(len(common_words) / len(expected_words) * 100, 1) if expected_words else 0

    # 4. Citation check (does it cite sources?)
    has_citation = any(c in rag_answer for c in ["[1]", "[2]", "[3]", "PMID"])
    scores["citations"] = 100 if has_citation else 0

    # 5. Hallucination check (does it say "bilgi yok" when it should?)
    # This is more complex - we'll implement later

    # Overall score (weighted average)
    weights = {"retrieval": 0.3, "similarity": 0.4, "key_terms": 0.2, "citations": 0.1}
    overall = sum(scores[k] * weights[k] for k in weights)
    scores["overall"] = round(overall, 1)

    return scores

# Test with a sample
print("✅ Evaluation function ready")

In [ ]:
# ============================================================
# 3. COLLECT RAG ANSWERS FOR EACH GROUND TRUTH QUESTION
# ============================================================

# Function to get RAG answer for a question
def get_rag_answer(question, top_k=3):
    """Get answer from our RAG system"""
    result = run_hybrid(question, top_k=top_k)
    hybrid_results = result["hybrid"]

    retrieved_pmids = [pmid for pmid, _ in hybrid_results]

    # Build context (same as before)
    retrieved_docs = []
    for pmid in retrieved_pmids:
        for meta in doc_metadata:
            if meta['pmid'] == pmid:
                retrieved_docs.append(meta)
                break

    context = ""
    for i, doc in enumerate(retrieved_docs, 1):
        context += f"\n--- DOCUMENT {i} ---\n"
        context += f"Title: {doc['title']}\n"
        context += f"Abstract: {doc['abstract'][:400]}\n"

    prompt = f"""Answer based ONLY on the context below. Be concise.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    # Call Gemini
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent?key={GEMINI_API_KEY}"
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.2, "maxOutputTokens": 200}
    }

    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            result_json = response.json()
            answer = result_json['candidates'][0]['content']['parts'][0]['text']
        else:
            answer = f"Error: {response.status_code}"
    except Exception as e:
        answer = f"Error: {e}"

    return answer, retrieved_pmids

In [ ]:
# ============================================================
# 4. RUN EVALUATION ON ALL QUESTIONS
# ============================================================

import time

print("=" * 70)
print("🎯 RUNNING GROUND TRUTH EVALUATION")
print("=" * 70)

evaluation_results = []

for q in all_questions:
    print(f"\n📝 Question {q['id']}: {q['question'][:50]}...")

    # Get RAG answer
    rag_answer_text, retrieved_pmids = get_rag_answer(q['question'], top_k=3)

    # Evaluate
    scores = evaluate_rag_answer(
        rag_answer_text,
        q['expected_answer'],
        retrieved_pmids,
        q['source_pmid']
    )

    evaluation_results.append({
        "id": q['id'],
        "term": q['term'],
        "question": q['question'],
        "expected_answer": q['expected_answer'],
        "rag_answer": rag_answer_text,
        "expected_pmid": q['source_pmid'],
        "retrieved_pmids": retrieved_pmids,
        "scores": scores
    })

    print(f"   Retrieval: {scores['retrieval']:.0f}% | Similarity: {scores['similarity']:.0f}% | Overall: {scores['overall']:.0f}%")

    # Rate limit delay
    time.sleep(3)

print("\n" + "=" * 70)
print("✅ EVALUATION COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# 5. SUMMARY REPORT
# ============================================================

print("\n" + "=" * 70)
print("📊 EVALUATION SUMMARY")
print("=" * 70)

# Aggregate scores by term
term_scores = {}
for result in evaluation_results:
    term = result['term']
    if term not in term_scores:
        term_scores[term] = {"retrieval": [], "similarity": [], "overall": []}

    term_scores[term]["retrieval"].append(result['scores']['retrieval'])
    term_scores[term]["similarity"].append(result['scores']['similarity'])
    term_scores[term]["overall"].append(result['scores']['overall'])

print(f"\n{'Term':<35} {'Retrieval':<12} {'Similarity':<12} {'Overall':<12}")
print("-" * 75)
for term, scores in term_scores.items():
    print(f"{term:<35} {np.mean(scores['retrieval']):>6.1f}%      {np.mean(scores['similarity']):>6.1f}%      {np.mean(scores['overall']):>6.1f}%")

# Overall averages
all_retrieval = np.mean([r['scores']['retrieval'] for r in evaluation_results])
all_similarity = np.mean([r['scores']['similarity'] for r in evaluation_results])
all_overall = np.mean([r['scores']['overall'] for r in evaluation_results])

print("\n" + "=" * 70)
print("🏆 FINAL SCORES")
print("=" * 70)
print(f"   Retrieval Accuracy (Hit@3): {all_retrieval:.1f}%")
print(f"   Answer Similarity:          {all_similarity:.1f}%")
print(f"   Overall RAG Quality:        {all_overall:.1f}%")

In [ ]:
# ============================================================
# 6. DETAILED RESULTS TABLE
# ============================================================

import pandas as pd

# Create DataFrame for detailed view
results_df = pd.DataFrame([
    {
        "ID": r['id'],
        "Term": r['term'],
        "Question": r['question'][:60] + "...",
        "Retrieval": r['scores']['retrieval'],
        "Similarity": r['scores']['similarity'],
        "Overall": r['scores']['overall'],
        "Expected PMID": r['expected_pmid'],
        "Retrieved PMIDs": str(r['retrieved_pmids'])
    }
    for r in evaluation_results
])

print("\n" + "=" * 70)
print("📋 DETAILED RESULTS TABLE")
print("=" * 70)
print(results_df.to_string(index=False))

In [ ]:
# ============================================================
# KONTROL: Beklenen makaleler corpus'ta var mı?
# ============================================================

print("=" * 70)
print("🔍 GROUND TRUTH MAKALELERİNİN CORPUS KONTROLÜ")
print("=" * 70)

# Corpus'taki tüm PMID'leri al
corpus_pmids = set(doc_ids)

# Ground truth'daki beklenen PMID'ler
expected_pmids = [
    "28265666",  # Atrial fibrillation
    "16326228",  # Atrial fibrillation management
    "40436475",  # AF in young
    "29229215",  # Neurogenic AF
    "30561633",  # AF ablation
    "34251351",  # T2DM cognitive
    "36967777",  # T2DM prevention
    "35242721",  # Gut microbiota T2DM
    "40416523",  # T2DM mechanisms
    "28707578",  # AOM children
    "24134083",  # AOM treatment
    "24439877",  # AOM bacteria
    "26784474",  # Celiac gold standard
    "33878915",  # Celiac serology
    "40999951",  # Celiac no-biopsy
    "31210940",  # Celiac treatment
]

print(f"\n📊 Corpus'taki makale sayısı: {len(corpus_pmids)}")
print(f"\n📋 Beklenen PMID'lerin durumu:")
print("-" * 50)

missing_pmids = []
present_pmids = []

for pmid in expected_pmids:
    if pmid in corpus_pmids:
        present_pmids.append(pmid)
        print(f"   ✅ {pmid} - VAR")
    else:
        missing_pmids.append(pmid)
        print(f"   ❌ {pmid} - YOK")

print(f"\n📊 ÖZET:")
print(f"   Toplam beklenen makale: {len(expected_pmids)}")
print(f"   Corpus'ta OLAN: {len(present_pmids)}")
print(f"   Corpus'ta OLMAYAN: {len(missing_pmids)}")

if missing_pmids:
    print(f"\n⚠️ SORUNLU OLANLAR (corpus'ta yok):")
    for pmid in missing_pmids:
        print(f"   - PMID {pmid}")

In [ ]:
# ============================================================
# SORUNLU SORGULARIN DETAYLI ANALİZİ
# ============================================================

print("=" * 70)
print("🔍 NEDEN BEKLENEN MAKALE RETRIEVAL'DE YOK?")
print("=" * 70)

problem_cases = [
    {
        "question": "What is atrial fibrillation?",
        "expected_pmid": "28265666",
        "expected_title": "Atrial Fibrillation."
    },
    {
        "question": "What are the common bacterial causes of acute otitis media?",
        "expected_pmid": "24439877",
        "expected_title": "Acute otitis media."
    },
    {
        "question": "What is the treatment for celiac disease?",
        "expected_pmid": "31210940",
        "expected_title": "European Society for the Study of Coeliac Disease (ESsCD) guideline..."
    }
]

for case in problem_cases:
    print(f"\n{'='*70}")
    print(f"📌 SORGU: {case['question']}")
    print(f"   BEKLENEN MAKALE: PMID {case['expected_pmid']}")
    print(f"   BAŞLIK: {case['expected_title'][:60]}...")

    # 1. Beklenen makalenin abstract'ını göster
    expected_article = None
    for meta in doc_metadata:
        if meta['pmid'] == case['expected_pmid']:
            expected_article = meta
            break

    if expected_article:
        print(f"\n   📄 BEKLENEN MAKALENİN ABSTRACT'ı:")
        print(f"      {expected_article['abstract'][:300]}...")

    # 2. Hybrid search ile ne geldiğini göster
    result = run_hybrid(case['question'], top_k=5)
    hybrid_results = result["hybrid"]

    print(f"\n   🔄 HYBRID RRF İLE GELENLER (top 5):")
    for i, (pmid, score) in enumerate(hybrid_results[:5], 1):
        for meta in doc_metadata:
            if meta['pmid'] == pmid:
                is_expected = "✅ BEKLENEN" if pmid == case['expected_pmid'] else ""
                print(f"      {i}. PMID {pmid}: {meta['title'][:50]}... {is_expected}")
                break

    # 3. BM25 skorlarına bak
    bm25_scores = bm25.get_scores(case['question'].lower().split())
    bm25_top = [(doc_ids[i], bm25_scores[i]) for i in np.argsort(bm25_scores)[::-1][:5]]

    print(f"\n   📊 BM25 İLE GELENLER:")
    for i, (pmid, score) in enumerate(bm25_top, 1):
        for meta in doc_metadata:
            if meta['pmid'] == pmid:
                is_expected = "✅ BEKLENEN" if pmid == case['expected_pmid'] else ""
                print(f"      {i}. PMID {pmid} (score={score:.4f}): {meta['title'][:40]}... {is_expected}")
                break

    # 4. Semantic skorlarına bak
    q_emb = model.encode(f"query: {case['question']}", normalize_embeddings=True)
    sem_scores = doc_embeddings @ q_emb
    sem_top = [(doc_ids[i], sem_scores[i]) for i in np.argsort(sem_scores)[::-1][:5]]

    print(f"\n   🧠 SEMANTIC İLE GELENLER:")
    for i, (pmid, score) in enumerate(sem_top, 1):
        for meta in doc_metadata:
            if meta['pmid'] == pmid:
                is_expected = "✅ BEKLENEN" if pmid == case['expected_pmid'] else ""
                print(f"      {i}. PMID {pmid} (sim={score:.4f}): {meta['title'][:40]}... {is_expected}")
                break

##  FINAL SYSTEM



In [ ]:
# ============================================================
# FINAL SYSTEM - HYBRID RRF + GEMINI RAG (Hit@5)
# ============================================================

import json
import requests
import numpy as np
import time
from difflib import SequenceMatcher
from google.colab import userdata

print("=" * 80)
print("🚀 FINAL RAG SYSTEM - HYBRID RRF + GEMINI 3.1 FLASH LITE")
print("=" * 80)

# ============================================================
# 1. LOAD CORPUS AND MODELS
# ============================================================

# Load corpus
with open("pubmed_corpus_final.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"✅ Loaded {len(corpus)} articles")

# Prepare documents for retrieval
documents = []
doc_ids = []
doc_metadata = []

for article in corpus:
    text = f"{article['title']} {article['abstract']}"
    documents.append(text)
    doc_ids.append(article['pmid'])
    doc_metadata.append({
        "pmid": article['pmid'],
        "title": article['title'],
        "abstract": article['abstract'],
        "journal": article['journal'],
        "year": article['year'],
        "query_terms": article['query_terms']
    })

print(f"✅ Prepared {len(documents)} documents")

# BM25 model (k1=1.5, b=0.75)
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs, k1=1.5, b=0.75)
print(f"✅ BM25 model ready (k1=1.5, b=0.75)")

# Semantic model
model = SentenceTransformer('intfloat/multilingual-e5-small')
doc_embeddings = model.encode(documents, normalize_embeddings=True, show_progress_bar=False)
print(f"✅ Semantic model ready (multilingual-e5-small, dim={doc_embeddings.shape[1]})")

# Gemini API
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
print(f"✅ Gemini API ready")

# ============================================================
# 2. RRF FUSION FUNCTION
# ============================================================

def run_hybrid(query, top_k=5, k_rrf=60):
    """Hybrid search using BM25 + Semantic + RRF"""

    # BM25
    bm25_sc = bm25.get_scores(query.lower().split())
    bm25_ranked = list(np.argsort(bm25_sc)[::-1])

    # Semantic
    q_emb = model.encode(f"query: {query}", normalize_embeddings=True)
    sem_sc = doc_embeddings @ q_emb
    sem_ranked = list(np.argsort(sem_sc)[::-1])

    # RRF fusion
    scores = {}
    for rank, doc_idx in enumerate(bm25_ranked, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k_rrf + rank)
    for rank, doc_idx in enumerate(sem_ranked, start=1):
        scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k_rrf + rank)

    rrf = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return {
        "bm25": [(doc_ids[i], bm25_sc[i]) for i in bm25_ranked[:top_k]],
        "semantic": [(doc_ids[i], sem_sc[i]) for i in sem_ranked[:top_k]],
        "hybrid": [(doc_ids[i], s) for i, s in rrf[:top_k]],
    }

# ============================================================
# 3. RAG ANSWER FUNCTION
# ============================================================

def rag_answer(query, top_k=3):
    """Generate cited answer using Hybrid RRF + Gemini"""

    result = run_hybrid(query, top_k=top_k)
    hybrid_results = result["hybrid"]

    retrieved_docs = []
    for pmid, score in hybrid_results:
        for meta in doc_metadata:
            if meta['pmid'] == pmid:
                retrieved_docs.append(meta)
                break

    context = ""
    sources = []
    for i, doc in enumerate(retrieved_docs, 1):
        context += f"\n--- DOCUMENT {i} ---\n"
        context += f"Title: {doc['title']}\n"
        context += f"Journal: {doc['journal']}, Year: {doc['year']}\n"
        context += f"Abstract: {doc['abstract'][:400]}\n"
        sources.append(f"[{i}] PMID {doc['pmid']}")

    prompt = f"""You are a medical AI assistant. Answer based ONLY on the context below. Cite sources with [1], [2]. Answer in the same language as the question.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent?key={GEMINI_API_KEY}"
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.2, "maxOutputTokens": 500}
    }

    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            result_json = response.json()
            answer = result_json['candidates'][0]['content']['parts'][0]['text']
        else:
            answer = f"Error: {response.status_code}"
    except Exception as e:
        answer = f"Error: {e}"

    return answer, sources, retrieved_docs

# ============================================================
# 4. EVALUATION WITH Hit@5
# ============================================================

def evaluate_rag_answer_hit5(rag_answer, expected_answer, retrieved_pmids, expected_pmid):
    """Evaluate RAG answer - Hit@5"""
    scores = {}

    # Hit@5: ilk 5 sonuca bakar
    if expected_pmid in retrieved_pmids[:5]:
        scores["retrieval"] = 100
        rank = retrieved_pmids.index(expected_pmid) + 1
        print(f"      ✅ Beklenen makale {rank}. sırada bulundu (Hit@5)")
    else:
        scores["retrieval"] = 0
        print(f"      ❌ Beklenen makale ilk 5'te yok")

    similarity = SequenceMatcher(None, rag_answer.lower()[:200], expected_answer.lower()[:200]).ratio()
    scores["similarity"] = round(similarity * 100, 1)

    expected_words = set(expected_answer.lower().split())
    rag_words = set(rag_answer.lower().split())
    common_words = expected_words & rag_words
    scores["key_terms"] = round(len(common_words) / len(expected_words) * 100, 1) if expected_words else 0

    has_citation = any(c in rag_answer for c in ["[1]", "[2]", "[3]", "PMID"])
    scores["citations"] = 100 if has_citation else 0

    weights = {"retrieval": 0.3, "similarity": 0.4, "key_terms": 0.2, "citations": 0.1}
    overall = sum(scores[k] * weights[k] for k in weights)
    scores["overall"] = round(overall, 1)

    return scores

# ============================================================
# 5. LOAD GROUND TRUTH
# ============================================================

with open("ground_truth.json", "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

all_questions = []
idx = 0
for term, questions in ground_truth["ground_truth"].items():
    for q in questions:
        all_questions.append({
            "id": idx,
            "term": term,
            "question": q["question"],
            "expected_answer": q["expected_answer"],
            "source_pmid": q["source_pmid"],
        })
        idx += 1

print(f"\n✅ Loaded {len(all_questions)} ground truth questions")
print(f"   Evaluation metric: Hit@5 (looks at top 5 results)")

# ============================================================
# 6. RUN EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("📊 RUNNING EVALUATION (Hit@5)")
print("=" * 80)

evaluation_results = []
success_count = 0

for q in all_questions:
    print(f"\n📝 Q{q['id']}: {q['question'][:55]}...")

    # Get RAG answer and retrieval
    rag_answer_text, sources, docs = rag_answer(q['question'], top_k=5)
    retrieved_pmids = [doc['pmid'] for doc in docs]

    # Evaluate with Hit@5
    scores = evaluate_rag_answer_hit5(
        rag_answer_text,
        q['expected_answer'],
        retrieved_pmids,
        q['source_pmid']
    )

    if scores['retrieval'] == 100:
        success_count += 1

    evaluation_results.append({
        "id": q['id'],
        "term": q['term'],
        "question": q['question'],
        "retrieved_pmids": retrieved_pmids,
        "expected_pmid": q['source_pmid'],
        "scores": scores
    })

    print(f"      Retrieval: {scores['retrieval']:.0f}% | Similarity: {scores['similarity']:.0f}% | Overall: {scores['overall']:.0f}%")

    time.sleep(2)  # Rate limit delay

# ============================================================
# 7. RESULTS SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("📊 EVALUATION SUMMARY (Hit@5)")
print("=" * 80)

# Per-term scores
term_scores = {}
for result in evaluation_results:
    term = result['term']
    if term not in term_scores:
        term_scores[term] = {"retrieval": [], "similarity": [], "overall": []}
    term_scores[term]["retrieval"].append(result['scores']['retrieval'])
    term_scores[term]["similarity"].append(result['scores']['similarity'])
    term_scores[term]["overall"].append(result['scores']['overall'])

print(f"\n{'Term':<35} {'Retrieval':<12} {'Similarity':<12} {'Overall':<12}")
print("-" * 75)
for term, scores in term_scores.items():
    print(f"{term:<35} {np.mean(scores['retrieval']):>6.1f}%      {np.mean(scores['similarity']):>6.1f}%      {np.mean(scores['overall']):>6.1f}%")

# Overall
all_retrieval = np.mean([r['scores']['retrieval'] for r in evaluation_results])
all_similarity = np.mean([r['scores']['similarity'] for r in evaluation_results])
all_overall = np.mean([r['scores']['overall'] for r in evaluation_results])

print("\n" + "=" * 80)
print("🏆 FINAL SCORES (Hit@5)")
print("=" * 80)
print(f"   Total questions: {len(evaluation_results)}")
print(f"   Successful retrieval (Hit@5): {success_count}/{len(evaluation_results)} ({success_count/len(evaluation_results)*100:.0f}%)")
print(f"   Retrieval Accuracy: {all_retrieval:.1f}%")
print(f"   Answer Similarity:  {all_similarity:.1f}%")
print(f"   Overall RAG Quality: {all_overall:.1f}%")

# Failed questions
failed = [r for r in evaluation_results if r['scores']['retrieval'] == 0]
if failed:
    print(f"\n⚠️ Failed questions ({len(failed)}):")
    for f in failed:
        print(f"   - Q{f['id']}: {f['question'][:50]}...")
else:
    print(f"\n✅ ALL QUESTIONS PASSED! (0 failed)")

print("\n" + "=" * 80)
print("✅ FINAL SYSTEM EVALUATION COMPLETE")
print("=" * 80)

In [ ]:
from google.colab import files

# İndirilecek dosyalar
files_to_download = [
    'pubmed_corpus_final.json',
    'medical_terms.csv',
    'ground_truth.json'
]

for file in files_to_download:
    try:
        files.download(file)
        print(f"✅ {file} indirildi")
    except:
        print(f"❌ {file} bulunamadı")